# 03 · Modelling

| Section | Models |
|---|---|
| **A. Baselines** | Naïve/Persistence · ARIMA(1,0,1) · GARCH(1,1) |
| **B. ML Models** | LightGBM · CatBoost |
| **C. Deep Learning** | TCN · PatchTST *(stubs — activate after DL env confirmed)* |
| **D. Regime Discovery** | GMM · HMM |
| **E. Anomaly & Change-Point** | Isolation Forest · Matrix Profile · CUSUM |

> ⚠️ Scaler/preprocessor always fitted on **training partition only**, applied unchanged to val/test.

**Reads:** `processed/train.parquet` · `processed/val.parquet` · `processed/test.parquet` · `processed/split_meta.yaml`  
**Saves:** `artifacts/models/*.pkl` · `artifacts/scalers/*.pkl` · `processed/metrics_draft.yaml` · `reports/figures/*.html`

In [ ]:
## Setup

import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ.pop("SPARK_HOME", None)   # Let pyspark 4.2.0 use its own bundled JARs

import sys, warnings, yaml, joblib
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baselines       import naive_persistence, fit_arima, fit_garch
from src.models.lgbm_model      import train_lgbm
from src.models.catboost_model  import train_catboost
from src.models.regime_models   import fit_gmm, fit_hmm
from src.models.anomaly_models  import fit_isolation_forest, compute_matrix_profile, detect_changepoints_cusum
from src.visualisation.model_plots  import plot_forecast_vs_actual, plot_feature_importance
from src.visualisation.regime_plots import plot_regime_states, plot_volatility_regimes

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
MODELS    = PROJECT_ROOT / CFG['artifacts']['models_dir']
SCALERS   = PROJECT_ROOT / CFG['artifacts']['scalers_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
METRICS   = PROJECT_ROOT / CFG['reports']['metrics_dir']
for d in [MODELS, SCALERS, FIGURES, METRICS]:
    d.mkdir(parents=True, exist_ok=True)

with open(PROCESSED / 'split_meta.yaml') as f:
    SPLIT = yaml.safe_load(f)

FEATURE_COLS = SPLIT['feature_cols']
TARGET_COL   = SPLIT['target_col']
TS_COL       = SPLIT['ts_col']

print(f'Features : {len(FEATURE_COLS)}  |  Target : {TARGET_COL}')

In [ ]:
# ── Load partitions ───────────────────────────────────────────────────
train = pd.read_parquet(PROCESSED / 'train.parquet')
val   = pd.read_parquet(PROCESSED / 'val.parquet')
test  = pd.read_parquet(PROCESSED / 'test.parquet')

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

print(f'Train {X_train.shape}  |  Val {X_val.shape}  |  Test {X_test.shape}')

In [ ]:
# ── Shared metric function ────────────────────────────────────────────
RESULTS = []  # collects all model results for notebook 04

def compute_metrics(y_true, y_pred, model='', partition=''):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    smape = float(np.mean(np.where(denom==0, 0, np.abs(y_true-y_pred)/denom)) * 100)
    result = {'model': model, 'partition': partition,
              'mae': mae, 'rmse': rmse, 'smape': smape}
    RESULTS.append(result)
    print(f'  [{model}] [{partition}]  MAE={mae:.6f}  RMSE={rmse:.6f}  sMAPE={smape:.2f}%')
    return result

---
## A · Baselines
### A1 · Naïve / Persistence

In [ ]:
# Predict: last observed return value
naive_val  = naive_persistence(train[TARGET_COL], horizon=1).iloc[-len(y_val):].fillna(0).values
naive_test = naive_persistence(
    pd.concat([train[TARGET_COL], val[TARGET_COL]]), horizon=1
).iloc[-len(y_test):].fillna(0).values

compute_metrics(y_val,  naive_val[:len(y_val)],   model='Naive', partition='Val')
compute_metrics(y_test, naive_test[:len(y_test)],  model='Naive', partition='Test')

### A2 · ARIMA(1,0,1)

In [ ]:
arima_series = train['log_return'].dropna()
print(f'Fitting ARIMA(1,0,1) on {len(arima_series):,} training obs...')
arima_res = fit_arima(arima_series, order=(1,0,1), forecast_steps=len(y_val)+len(y_test))
print(f'AIC={arima_res["aic"]:.2f}  BIC={arima_res["bic"]:.2f}')

fc = arima_res['forecast'].values
compute_metrics(y_val,  fc[:len(y_val)],                      model='ARIMA(1,0,1)', partition='Val')
compute_metrics(y_test, fc[len(y_val):len(y_val)+len(y_test)], model='ARIMA(1,0,1)', partition='Test')

### A3 · GARCH(1,1)

In [ ]:
garch_series = train['log_return'].dropna()
print(f'Fitting GARCH(1,1) on {len(garch_series):,} training obs...')
garch_res = fit_garch(garch_series,
                      p=CFG['models']['garch']['p'],
                      q=CFG['models']['garch']['q'],
                      dist=CFG['models']['garch']['dist'],
                      forecast_horizon=1)
print(f'AIC={garch_res["aic"]:.2f}  BIC={garch_res["bic"]:.2f}')
print(f'1-step vol forecast (annualised): {float(garch_res["variance_forecast"][0])*np.sqrt(365):.4f}')

# Save conditional vol for Streamlit
cond_vol = garch_res['conditional_vol']
cv_index = train['log_return'].dropna().index
cv_df = pd.DataFrame({TS_COL: train.loc[cv_index, TS_COL].values,
                      'cond_vol': cond_vol.values[:len(cv_index)]})
cv_df.to_csv(MODELS / 'garch_cond_vol.csv', index=False)
print('GARCH conditional vol saved.')

---
## B · ML Models
### B1 · LightGBM

In [ ]:
print('Training LightGBM...')
lgbm_res   = train_lgbm(X_train, y_train, X_val, y_val,
                         task='regression', artifact_name='lgbm_return')
lgbm_model  = lgbm_res['model']
lgbm_scaler = lgbm_res['scaler']

lgbm_val_pred  = lgbm_model.predict(lgbm_scaler.transform(X_val))
lgbm_test_pred = lgbm_model.predict(lgbm_scaler.transform(X_test))
compute_metrics(y_val,  lgbm_val_pred,  model='LightGBM', partition='Val')
compute_metrics(y_test, lgbm_test_pred, model='LightGBM', partition='Test')

fig = plot_forecast_vs_actual(y_test, pd.Series(lgbm_test_pred, index=y_test.index),
                              index=test.loc[y_test.index, TS_COL],
                              title='LightGBM — Test Forecast vs Actual')
fig.write_html(str(FIGURES / 'lgbm_forecast.html'))
fig.show()

fig_imp = plot_feature_importance(FEATURE_COLS, lgbm_model.feature_importances_,
                                  top_n=25, title='LightGBM — Feature Importance')
fig_imp.write_html(str(FIGURES / 'lgbm_feature_importance.html'))
fig_imp.show()

### B2 · CatBoost

In [ ]:
print('Training CatBoost...')
cb_res   = train_catboost(X_train, y_train, X_val, y_val,
                           task='regression', artifact_name='catboost_return')
cb_model  = cb_res['model']
cb_scaler = cb_res['scaler']

cb_val_pred  = cb_model.predict(cb_scaler.transform(X_val))
cb_test_pred = cb_model.predict(cb_scaler.transform(X_test))
compute_metrics(y_val,  cb_val_pred,  model='CatBoost', partition='Val')
compute_metrics(y_test, cb_test_pred, model='CatBoost', partition='Test')

fig = plot_forecast_vs_actual(y_test, pd.Series(cb_test_pred, index=y_test.index),
                              index=test.loc[y_test.index, TS_COL],
                              title='CatBoost — Test Forecast vs Actual')
fig.write_html(str(FIGURES / 'catboost_forecast.html'))
fig.show()

---
## C · Deep Learning — TCN & PatchTST
> **Stubs** — uncomment after verifying PyTorch / darts environment.
> TCN: local temporal features via dilated causal convolutions.  
> PatchTST: patch-based Transformer for time-series forecasting.

In [ ]:
# import torch
# from darts import TimeSeries
# from darts.models import TCNModel, PatchTST
#
# ts_train = TimeSeries.from_dataframe(train, time_col=TS_COL, value_cols='log_return')
# ts_val   = TimeSeries.from_dataframe(val,   time_col=TS_COL, value_cols='log_return')
# ts_test  = TimeSeries.from_dataframe(test,  time_col=TS_COL, value_cols='log_return')
#
# # ── TCN ──────────────────────────────────────────────────────────────
# tcn = TCNModel(input_chunk_length=60, output_chunk_length=1,
#                num_filters=32, kernel_size=3, dilation_base=2,
#                n_epochs=20, random_state=42)
# tcn.fit(ts_train, val_series=ts_val)
# tcn_pred = tcn.predict(len(ts_test))
# tcn.save(str(MODELS / 'tcn_model'))
#
# # ── PatchTST ─────────────────────────────────────────────────────────
# patchtst = PatchTST(input_chunk_length=60, output_chunk_length=1,
#                     d_model=64, nhead=4, num_encoder_layers=2,
#                     n_epochs=20, random_state=42)
# patchtst.fit(ts_train, val_series=ts_val)
# patchtst_pred = patchtst.predict(len(ts_test))
# patchtst.save(str(MODELS / 'patchtst_model'))

print('DL stubs ready. Uncomment above after verifying PyTorch/darts environment.')

---
## D · Regime Discovery
### D1 · GMM + HMM

In [ ]:
# ── Regime feature set (vol + return — good discriminators) ──────────
regime_fcols = [c for c in FEATURE_COLS
                if any(x in c for x in ('rolling_vol','log_return','momentum','rolling_return'))]
print(f'Regime features ({len(regime_fcols)}): {regime_fcols}')

X_reg_train = train[regime_fcols].dropna().values
X_reg_full  = pd.read_parquet(PROCESSED / 'btc_features.parquet')[regime_fcols].dropna().values

regime_scaler = StandardScaler()
X_reg_train_s = regime_scaler.fit_transform(X_reg_train)
X_reg_full_s  = regime_scaler.transform(X_reg_full)
joblib.dump(regime_scaler, SCALERS / 'regime_scaler.pkl')
print('Regime scaler fitted on training data only.')

In [ ]:
# ── GMM ───────────────────────────────────────────────────────────────
print('Fitting GMM...')
gmm_res    = fit_gmm(X_reg_train_s, artifact_name='gmm')
gmm_states = gmm_res['model'].predict(X_reg_full_s)

df_full = pd.read_parquet(PROCESSED / 'btc_features.parquet')
df_full[TS_COL] = pd.to_datetime(df_full[TS_COL])
regime_index    = df_full[regime_fcols].dropna().index
df_full.loc[regime_index, 'gmm_regime'] = gmm_states.astype(int)

print(pd.Series(gmm_states).value_counts().sort_index().rename('bars_in_regime'))

fig = plot_regime_states(df_full.loc[regime_index], ts_col=TS_COL,
                         state_col='gmm_regime', title='GMM Market Regimes')
fig.write_html(str(FIGURES / 'gmm_regimes.html'))
fig.show()

In [ ]:
# ── HMM ───────────────────────────────────────────────────────────────
print('Fitting HMM...')
hmm_res    = fit_hmm(X_reg_train_s, artifact_name='hmm')
hmm_states = hmm_res['model'].predict(X_reg_full_s)
df_full.loc[regime_index, 'hmm_state'] = hmm_states.astype(int)

print(pd.Series(hmm_states).value_counts().sort_index().rename('bars_in_state'))
print('\nHMM Transition Matrix:')
n = hmm_res['model'].n_components
print(pd.DataFrame(hmm_res['model'].transmat_,
                   columns=[f'→S{i}' for i in range(n)],
                   index=[f'S{i}→' for i in range(n)]).round(3))

fig = plot_regime_states(df_full.loc[regime_index], ts_col=TS_COL,
                         state_col='hmm_state', title='HMM Market States')
fig.write_html(str(FIGURES / 'hmm_states.html'))
fig.show()

---
## E · Anomaly & Change-Point Detection
### E1 · Isolation Forest

In [ ]:
print('Fitting Isolation Forest...')
X_anom_train = train[FEATURE_COLS].dropna().values
X_anom_full  = df_full[FEATURE_COLS].dropna().values

anom_scaler   = StandardScaler()
X_anom_tr_s   = anom_scaler.fit_transform(X_anom_train)
X_anom_full_s = anom_scaler.transform(X_anom_full)
joblib.dump(anom_scaler, SCALERS / 'anomaly_scaler.pkl')

iso_res = fit_isolation_forest(X_anom_tr_s, X_anom_full_s, artifact_name='isolation_forest')

anom_index = df_full[FEATURE_COLS].dropna().index
scores_df  = pd.DataFrame({
    TS_COL:       df_full.loc[anom_index, TS_COL].values,
    'score':      iso_res['full_scores'],
    'is_anomaly': iso_res['anomaly_flags'].astype(int),
})
scores_df.to_csv(MODELS / 'anomaly_scores.csv', index=False)
print(f'Anomalies flagged: {iso_res["anomaly_flags"].sum():,} '
      f'({iso_res["anomaly_flags"].mean()*100:.2f}%)')

fig = go.Figure([
    go.Scatter(x=pd.to_datetime(scores_df[TS_COL]), y=scores_df['score'],
               name='IF Score', line=dict(color='#4FC3F7', width=0.8)),
    go.Scatter(x=pd.to_datetime(scores_df[scores_df['is_anomaly']==1][TS_COL]),
               y=scores_df[scores_df['is_anomaly']==1]['score'],
               mode='markers', name='Anomaly',
               marker=dict(color='#EF5350', size=5, symbol='x')),
])
fig.update_layout(title='Isolation Forest — Anomaly Scores', template='plotly_dark')
fig.write_html(str(FIGURES / 'anomaly_scores.html'))
fig.show()

### E2 · Matrix Profile

In [ ]:
print('Computing Matrix Profile (may take a few minutes)...')

# Use training log_return series only (correct partition for unsupervised discovery).
# Cap at 50 000 bars — stumpy.stump() is O(n²); beyond this it becomes very slow.
MP_MAX_ROWS = 50_000
lr_series = train['log_return'].dropna().iloc[-MP_MAX_ROWS:].values
mp_index  = train['log_return'].dropna().iloc[-MP_MAX_ROWS:].index

print(f'  Series length : {len(lr_series):,} bars (window=30)')
mp_res   = compute_matrix_profile(lr_series, window=30)
disc_idx = mp_res['discord_idx']

disc_ts = train.loc[mp_index[disc_idx], TS_COL] if disc_idx < len(mp_index) else 'N/A'
print(f'Discord (most unusual 30-bar window) at index {disc_idx}  →  {disc_ts}')
print(f'Discord MP value: {mp_res["discord_value"]:.4f}')

mp_ts = pd.to_datetime(train.loc[mp_index[:len(mp_res["mp"])], TS_COL].values)
fig = go.Figure([go.Scatter(x=mp_ts, y=mp_res['mp'], name='MP', line=dict(color='#F7931A'))])
if disc_idx < len(mp_ts):
    fig.add_vline(x=mp_ts[disc_idx], line_color='#EF5350', line_dash='dot',
                  annotation_text='Discord')
fig.update_layout(title='Matrix Profile — Log Returns (window=30)', template='plotly_dark')
fig.write_html(str(FIGURES / 'matrix_profile.html'))
fig.show()

### E3 · CUSUM Change-Point Detection

In [ ]:
# ⚠️  Calibrated on training partition only
cusum_series = train['log_return'].dropna().values
print(f'Detecting change-points on {len(cusum_series):,} training observations...')

bkps = detect_changepoints_cusum(cusum_series, n_bkps=5, model='rbf')

train_ts_index = train['log_return'].dropna().index
bp_records = []
for bp in bkps[:-1]:
    if bp < len(train_ts_index):
        ts_val = train.loc[train_ts_index[bp], TS_COL]
        bp_records.append({'index': bp, 'timestamp': str(ts_val)})
        print(f'  Breakpoint {bp} → {ts_val}')

pd.DataFrame(bp_records).to_csv(MODELS / 'changepoints.csv', index=False)

vol_col = next((c for c in FEATURE_COLS if 'rolling_vol_' in c), 'log_return')
fig = plot_volatility_regimes(
    df_full.assign(**{TS_COL: pd.to_datetime(df_full[TS_COL])}),
    vol_col=vol_col, ts_col=TS_COL,
    changepoints=[r['index'] for r in bp_records],
    title='CUSUM Change-Points on Rolling Volatility',
)
fig.write_html(str(FIGURES / 'cusum_changepoints.html'))
fig.show()

In [ ]:
# ── Save interim metrics for notebook 04 ─────────────────────────────
metrics_path = PROCESSED / 'metrics_draft.yaml'
with open(metrics_path, 'w') as f:
    yaml.dump(RESULTS, f, default_flow_style=False)
print(f'Interim metrics saved → {metrics_path}')
print('Next → 04_evaluation_and_export.ipynb')